# 03 — Prophet Model Development

> **Portfolio Optimizer** · *Bayesian Structural Time Series Forecasting*

**Objectives**
1. Configure Prophet with NSE-specific seasonalities (budget_season, earnings_season, diwali, FII quarterly).
2. Add RSI, MACD, and Volume as external regressors.
3. Decompose trend, seasonality, and regressor components.
4. Cross-validate with walk-forward expanding window.
5. Visualise forecast components and regressor impacts.
6. Log experiments to MLflow.

In [ ]:
import warnings, os, sys
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
import mlflow

NAVY, SLATE, ORANGE, GREY = '#1A273A', '#3E4A62', '#C24D2C', '#D9D9D7'
def base_layout(**kw):
    return dict(paper_bgcolor=NAVY, plot_bgcolor=NAVY,
                font=dict(color=GREY), xaxis=dict(gridcolor=SLATE),
                yaxis=dict(gridcolor=SLATE), **kw)

TICKER = 'RELIANCE.NS'
print(f'Prophet version: {Prophet.__module__.split(".")[0]}  |  Ticker: {TICKER}')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
from data.data_ingestion import load_processed
from models.prophet_model import prepare_prophet_df

df = load_processed(TICKER)
prophet_df = prepare_prophet_df(df)
print(f'Prophet DataFrame shape: {prophet_df.shape}')
prophet_df.head(3)

In [ ]:
# ── Build and train Prophet model ─────────────────────────────────────────────
from models.prophet_model import build_prophet_model

model = build_prophet_model(
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10.0,
    uncertainty_samples=1000,
    interval_width=0.95,
)

# Fit on all data except last 90 days (hold-out)
train_df = prophet_df.iloc[:-90]
model.fit(train_df)
print('Model fitted successfully.')
print(f'Changepoints detected: {len(model.changepoints)}')

In [ ]:
# ── Generate 90-day forecast ──────────────────────────────────────────────────
future    = model.make_future_dataframe(periods=90, freq='B')
# Fill regressors for future dates with last known values
for col in ['RSI_14', 'MACD', 'Volume_normalized']:
    if col in prophet_df.columns:
        future[col] = future['ds'].map(
            prophet_df.set_index('ds')[col]
        ).fillna(prophet_df[col].iloc[-1])

forecast = model.predict(future)
print(f'Forecast shape: {forecast.shape}')
forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(5)

In [ ]:
# ── Plotly forecast visualisation ────────────────────────────────────────────
actual   = prophet_df.set_index('ds')['y']
hold_out = actual.iloc[-90:]
pred     = forecast.set_index('ds')

fig = go.Figure()
fig.add_trace(go.Scatter(x=actual.index, y=actual,
                          name='Actual', line=dict(color=GREY, width=1.5)))
fig.add_trace(go.Scatter(x=pred.index, y=pred['yhat'],
                          name='Forecast', line=dict(color=ORANGE, width=2, dash='dash')))
fig.add_trace(go.Scatter(
    x=pd.concat([pred.index.to_series(), pred.index.to_series()[::-1]]),
    y=pd.concat([pred['yhat_upper'], pred['yhat_lower'][::-1]]),
    fill='toself', fillcolor='rgba(194,77,44,0.12)',
    line=dict(color='rgba(0,0,0,0)'), name='95% CI',
))

fig.update_layout(**base_layout(title=f'{TICKER} — Prophet Forecast + 95% CI', height=500))
fig.show()

In [ ]:
# ── Decompose trend + seasonalities ──────────────────────────────────────────
components = ['trend', 'yearly', 'weekly']
fig = make_subplots(rows=len(components), cols=1,
    subplot_titles=[c.title() for c in components])

colors = [ORANGE, GREY, SLATE]
for i, comp in enumerate(components):
    if comp in forecast.columns:
        fig.add_trace(
            go.Scatter(x=forecast['ds'], y=forecast[comp],
                       name=comp.title(), line=dict(color=colors[i], width=1.5)),
            row=i+1, col=1,
        )

fig.update_layout(**base_layout(title='Prophet Decomposition', height=700))
for i in range(1, len(components)+1):
    fig.update_xaxes(gridcolor=SLATE, row=i, col=1)
    fig.update_yaxes(gridcolor=SLATE, row=i, col=1)
fig.show()

In [ ]:
# ── Walk-forward cross-validation ────────────────────────────────────────────
print('Running cross-validation (this may take a few minutes)...')
try:
    df_cv = cross_validation(
        model, initial='730 days', period='90 days', horizon='30 days',
        parallel='processes',
    )
    df_perf = performance_metrics(df_cv)
    print('\nCross-validation performance metrics:')
    print(df_perf[['horizon','mse','rmse','mape','coverage']].round(4).to_string(index=False))

    # Plot MAPE vs horizon
    fig = go.Figure(go.Scatter(
        x=df_perf['horizon'].dt.days, y=df_perf['mape'] * 100,
        mode='lines+markers', line=dict(color=ORANGE, width=2),
        fill='tozeroy', fillcolor='rgba(194,77,44,0.15)',
    ))
    fig.update_layout(**base_layout(title='Prophet MAPE vs Forecast Horizon', height=400,
                                      xaxis_title='Horizon (days)', yaxis_title='MAPE (%)'))
    fig.show()
except Exception as e:
    print(f'Cross-validation skipped: {e}')

In [ ]:
# ── Changepoint visualisation ─────────────────────────────────────────────────
cps = model.changepoints
fig = go.Figure()
fig.add_trace(go.Scatter(x=actual.index, y=actual,
                          name='Close', line=dict(color=GREY, width=1.5)))
for cp in cps:
    fig.add_vline(x=cp, line=dict(color=ORANGE, width=1, dash='dot'), opacity=0.6)

fig.update_layout(**base_layout(
    title=f'{TICKER} — Prophet Detected Changepoints ({len(cps)} total)', height=450,
))
fig.show()
print(f'\n✅  Prophet Development complete for {TICKER}')